# Student Performance Prediction: Exploratory Data Analysis (EDA)

## 1. Project Background and Objective
This project focuses on predicting student academic performance. Specifically, our objective is to predict the **Math Score** of students based on various demographic factors and academic indicators. 

Conducting a thorough Exploratory Data Analysis (EDA) allows us to:
1. Understand the distribution, central tendencies, and spreads of assessment scores.
2. Examine the demographics of the student body (Gender, Race, Parental Education, Lunch type, Test Prep course).
3. Identify relationships, correlations, and multivariate interactions that influence performance.
4. Prepare insights that will guide preprocessing and model-building decisions (e.g., standard scaling, encoding categorical variables).

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

## 2. Ingest and Clean Dataset

We start by loading the dataset. To ensure standard Python variable naming conventions later, we will clean the column headers by replacing spaces and slashes with underscores.

In [2]:
df = pd.read_csv('../data/stud.csv')
# Standardize column headers
df.columns = [col.strip().replace(' ', '_').replace('/', '_') for col in df.columns]
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


## 3. Structural Data Exploration

Let's inspect the shapes, types, missing values, duplicates, and cardinality of each column in the dataset to verify data quality.

In [3]:
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns\n")
df.info()

Dataset Dimensions: 1000 rows, 8 columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   gender                       1000 non-null   object
 1   race_ethnicity               1000 non-null   object
 2   parental_level_of_education  1000 non-null   object
 3   lunch                        1000 non-null   object
 4   test_preparation_course      1000 non-null   object
 5   math_score                   1000 non-null   int64 
 6   reading_score                1000 non-null   int64 
 7   writing_score                1000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 62.6+ KB


### Data Quality Checks (Nulls and Duplicates)
Missing values can degrade model quality, while duplicate entries can skew distributions and cause data leakage during training splits.

In [4]:
print("Missing Values Per Feature:")
print(df.isna().sum())
print(f"\nDuplicate rows found: {df.duplicated().sum()}")

Missing Values Per Feature:
gender                         0
race_ethnicity                 0
parental_level_of_education    0
lunch                          0
test_preparation_course        0
math_score                     0
reading_score                  0
writing_score                  0
dtype: int64

Duplicate rows found: 0


### Exploring Unique Values and Feature Types
We separate the columns into categorical and numerical, listing the distinct groups for each categorical feature.

In [5]:
numeric_features = [feature for feature in df.columns if df[feature].dtype != 'O']
categorical_features = [feature for feature in df.columns if df[feature].dtype == 'O']

print(f"Numerical Features: {numeric_features}")
print(f"Categorical Features: {categorical_features}\n")

for col in categorical_features:
    print(f"Unique categories in '{col}': {df[col].unique()}")

Numerical Features: ['math_score', 'reading_score', 'writing_score']
Categorical Features: ['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course']

Unique categories in 'gender': ['female' 'male']
Unique categories in 'race_ethnicity': ['group B' 'group C' 'group A' 'group D' 'group E']
Unique categories in 'parental_level_of_education': ["bachelor's degree" 'some college' "master's degree" "associate's degree"
 'high school' 'some high school']
Unique categories in 'lunch': ['standard' 'free/reduced']
Unique categories in 'test_preparation_course': ['none' 'completed']


## 4. Descriptive Summary Statistics

Let's look at the basic statistical summary (mean, standard deviation, percentiles, minimum, and maximum values) of numerical features.

In [ ]:
df.describe()

### Summary Observations:
1. All score features (`math_score`, `reading_score`, `writing_score`) are out of a maximum of 100.
2. Mean performance sits closely in the high 60s (~66.0 for math, ~69.1 for reading, ~68.0 for writing).
3. Standard deviation is around 14.6 - 15.2, indicating consistent variance across all assessment fields.
4. The minimum score for math is 0, indicating a potential outlier or non-attempt, while reading and writing minimums are higher (17 and 10).

## 5. Feature Engineering

To understand the student's overall performance, we engineer two aggregate features:
- `total_score`: Cumulative score of Math, Reading, and Writing.
- `average`: Average grade across all three assessments.

In [ ]:
df['total_score'] = df['math_score'] + df['reading_score'] + df['writing_score']
df['average'] = df['total_score'] / 3
df.head()

## 6. Visualizing Distributions & Insights

### A. Assessment Score Distributions
We analyze the distribution of student average scores and check if gender introduces any shift in performance.

In [ ]:
plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
sns.histplot(data=df, x='average', bins=30, kde=True, color='purple')
plt.title('Distribution of Student Average Scores')

plt.subplot(1, 2, 2)
sns.histplot(data=df, x='average', kde=True, hue='gender', palette='Set1')
plt.title('Student Average Score Distribution by Gender')
plt.tight_layout()
plt.show()

**Insight**: Average student score is normally distributed around ~68. When broken down by gender, both males and females follow a normal distribution, but female students have a slightly higher overall density in top average score ranges.

### B. Target vs Score Components
Let's check the relationship between our target variable `math_score` and the other assessment components (`reading_score` and `writing_score`).

In [ ]:
plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
sns.scatterplot(data=df, x='reading_score', y='math_score', hue='gender', alpha=0.7)
plt.title('Reading Score vs Math Score')

plt.subplot(1, 2, 2)
sns.scatterplot(data=df, x='writing_score', y='math_score', hue='gender', alpha=0.7)
plt.title('Writing Score vs Math Score')
plt.tight_layout()
plt.show()

**Insight**: There is a strong, linear correlation between math scores and reading/writing scores. This implies that academic aptitude is highly consistent across subjects. Students who do well in reading and writing are highly likely to score high marks in mathematics.

## 7. Categorical Feature Impact Analysis

We use box plots to compare the distributions of average scores across categories. Box plots help us visualize medians, quartiles, and highlight potential outliers.

In [ ]:
plt.figure(figsize=(18, 12))

plt.subplot(2, 2, 1)
sns.boxplot(data=df, x='lunch', y='average', palette='Set2')
plt.title('Influence of Lunch Type on Average Performance')

plt.subplot(2, 2, 2)
sns.boxplot(data=df, x='test_preparation_course', y='average', palette='Accent')
plt.title('Influence of Test Prep Course on Average Performance')

plt.subplot(2, 2, 3)
sns.boxplot(data=df, x='race_ethnicity', y='average', order=sorted(df['race_ethnicity'].unique()), palette='Pastel1')
plt.title('Influence of Race/Ethnicity on Average Performance')

plt.subplot(2, 2, 4)
sns.boxplot(data=df, y='parental_level_of_education', x='average', palette='coolwarm')
plt.title('Influence of Parental Level of Education')

plt.tight_layout()
plt.show()

### Key Demographic Insights:
1. **Lunch**: Students with standard lunches score significantly higher on average than students on free/reduced lunch, highlighting the link between socio-economic/nutritional support and academic results.
2. **Test Preparation**: Completing a test prep course raises student average performance. The median score is visibly higher for the 'completed' category.
3. **Race/Ethnicity**: Group E has the highest average performance median, followed by Group D and Group C, whereas Group A displays the lowest median.
4. **Parental Education**: Parents holding a Master's or Bachelor's degree have children with higher performance medians. High school education levels correlate with lower median scores.

## 8. Correlation Analysis

Let's build a correlation matrix to confirm the strength of numerical associations.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_features].corr(), annot=True, cmap='Blues', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

**Insight**: Reading and writing scores are extremely highly correlated (0.95), suggesting similar skill sets. Math score is also highly correlated with reading (0.82) and writing (0.80). This justifies using reading and writing scores as strong continuous predictors in our machine learning regression pipeline.